# Stage 1

## install
```
pip install ftfy bs4 trafilatura
pip install ipykernel 
```

## dataset
wget https://code.s3.yandex.net/ai-engineer/Итоговый_проект_спринт2/filmoskop_kb.json -O filmoskop_kb.json

In [1]:
'''
1. Load primary docs
'''
import json

with open("filmoskop_kb.json", encoding="utf-8") as f:
    data = json.load(f)
docs = data["docs"]        # 181 раздел базы знаний
queries = data["queries"]  # 66 тестовых запросов с разметкой

print("Разделов:", len(docs), "| Запросов:", len(queries))

Разделов: 181 | Запросов: 66


In [2]:
'''
1.1 count junk_ratio - pre
'''
import re

def junk_count(text):
  if not text:
      return 0.0
  # Мусор: всё, что не буква, не цифра, не пробел и не базовый знак препинания
  junk = re.findall(r"[^\w\s.,!?;:()«»—-]", text)
  return len(junk) 


text_len = 0
junk_len = 0
for t in docs:
  text_len += len(t['raw'])
  junk_len += junk_count(str(t['raw']))

print('Junk ratio: ', round(junk_len/text_len, 2), ' | Junk length: ', junk_len,' | Text length: ', text_len)

Junk ratio:  0.05  | Junk length:  7079  | Text length:  145549


In [5]:
'''
2. filter HTML
'''
from bs4 import BeautifulSoup  
import trafilatura

def normalize_html(raw):
  soup = BeautifulSoup(raw, "html.parser")
  for tag in soup(["script", "style", "nav", "footer"]):
      tag.decompose()
  for node in soup.select(".cookie"):
      node.decompose()
  return soup.get_text(" ")

for t in docs:
  if t['format'] == 'html':
    txt = trafilatura.extract(t['raw'], favor_precision=False, include_comments=False) 
    if not txt:
    #  print(t['doc_id'])
     txt = normalize_html(t['raw'])
    t['raw'] = txt

# filtered = [i for i in docs if i['raw']]

# write to file
with open("tmp/docs_html.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=4)

In [11]:
'''
3. encoding, normalize text
'''
import re
import unicodedata
import ftfy

def encode_text(text):
  # Если текст побит — восстанавливаем «кракозябры»
  return ftfy.fix_text(text)

def normalize_text(text):
  # Приводим юникод к единой форме NFC
  text = unicodedata.normalize("NFC", text)

  # Склеиваем слово, разорванное переносом строки
  text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

  # Неразрывные и нулевой ширины пробелы -> обычный пробел
  text = text.replace("\N{NO-BREAK SPACE}", " ")
  text = text.replace("\N{ZERO WIDTH SPACE}", " ")

  # Схлопываем подряд идущие пробелы и переносы в один пробел
  text = re.sub(r"\s+", " ", text)

  return text.strip()


for t in docs:
  if not t['raw']:
    print('>>>', t)
    continue
  if t['format'] != 'html':
    t['raw'] = encode_text(t['raw'])

In [9]:
'''
remove useless data and put period in the end of item
'''
MIN_LENGTH = 30

filtered = []
for i in docs:
  if len(i['raw']) < MIN_LENGTH \
  or 'Внимание: статья устарела' in i['raw'] \
  or 'выгружено автоматически' in i['raw'] \
  or 'Раздел временно недоступен' in i['raw'] \
  or 'Страница не найдена' in i['raw'] \
  or 'Черновик.' in i['raw'] \
  or 'черновик, не публиковать' in i['raw']:
    continue
  
  i['raw'] = i['raw'].replace("Статья была полезна? Да Нет", "").strip()
  # i['raw'] = i['raw'].replace("— Справка «Фильмоскопа» Главная → Справка →", "").strip()
  # i['raw'] = i['raw'].replace("Мы используем cookie, чтобы сервис работал лучше", "").strip()
  i['raw'] = i['raw'].replace("→", "-").strip()

  if not i['raw'].endswith("."):
    i['raw'] = f"{i['raw']}." 

  filtered.append(i)

In [ ]:
for t in filtered:
  t['raw'] = normalize_text(t['raw'])

with open("1_corpus.json", "w", encoding="utf-8") as f:
    json.dump(filtered, f, ensure_ascii=False, indent=4)

In [52]:
'''
3.1 count junk_ratio - post
'''
import re

def junk_count(text):
  if not text:
      return 0.0
  # Мусор: всё, что не буква, не цифра, не пробел и не базовый знак препинания
  junk = re.findall(r"[^\w\s.,!?;:()«»—-]", text)

  # print('>>> junk', junk)
  return len(junk) 


text_len = 0
junk_len = 0
for t in docs:
  text_len += len(t['raw'])
  junk_len += junk_count(str(t['raw']))

print('Junk ratio: ', round(junk_len/text_len, 2), ' | Junk length: ', junk_len,' | Text length: ', text_len)

Junk ratio:  0.0  | Junk length:  147  | Text length:  128446


In [13]:
'''
3.2 Check Duplicates
'''

from collections import Counter

# docs as a list
texts = [item["raw"] for item in docs]

counts = Counter(texts)
duplicates = [item for item, count in counts.items() if count > 1]
print('>>> Duplicates', duplicates)

>>> Duplicates []


In [54]:
MIN_LENGTH = 30

texts = [item["raw"] for item in filtered]

total = 0
total_length = 0
count = 0
for item in texts:
  total += 1
  total_length += len(item)
  if len(item) < MIN_LENGTH:
    count += 1
    print(item)

shortest = min(texts, key=len)
longest = max(texts, key=len)

print(
  '>>> suspicious lengths: ', count, 
  ' | Mean length: ', round(total_length/total, 2), 
  ' | min: ', len(shortest), 
  ' | longest: ', len(longest),
  ' | total docs: ', len(texts),
)
# print(longest)

>>> suspicious lengths:  0  | Mean length:  760.34  | min:  70  | longest:  1853  | total docs:  154


In [ ]:
'''
Normalize Queries
'''
for t in queries:
  t['query'] = normalize_text(t['query'])
  t['gold'] = normalize_text(t['gold'])

with open("1_queries.json", "w", encoding="utf-8") as f:
    json.dump(queries, f, ensure_ascii=False, indent=4)

# Stage 1 - Report

## Исходный датасет
Разделов: 181 | Запросов: 66 | Text length:  145549

Форматы:
- text
- html

Junk:
- Junk ratio:  0.05  
- Junk length:  7079  


## Transformations
-  HTML -> TEXT
- Encoding unification
- Text normalization
- Suspicious lengths
- Manual audit

## Suspicious lengths
Selected: MIN_LENGTH = 10
as there could be useful string like 'QX-500;2020;да'

## Junk ratio 
JUNK_THRESHOLD = 0.15

0.05 -> 0.0 (167 / 130429)
- acceptable

## Manual test
- acceptable


## Final result
1_corpus.json
1_queries.json